In [4]:
import os
import struct
import math
import numpy as np
import cv2          # hanya untuk cv2.imread
from tqdm import tqdm

# ============================================================
# KONFIGURASI
# ============================================================
PATH_INPUT  = "D:/semester 4/PCD/praktikum/projek/test/Assets/"
PATH_OUTPUT = "D:/semester 4/PCD/praktikum/projek/test/Assets_Prepro4/"
KATEGORI    = ["Normal", "kidneyStone"]


# ============================================================
# FUNGSI MANUAL PENGGANTI NumPy (tanpa fungsi yang dilarang)
# ============================================================

def manual_clip(arr, a_min, a_max):
    """Mengganti np.clip."""
    if isinstance(arr, np.ndarray):
        flat = arr.flat
        return np.array([max(a_min, min(a_max, x)) for x in flat]).reshape(arr.shape)
    else:
        return max(a_min, min(a_max, arr))

def manual_arange(start, stop=None, step=1):
    """Mengganti np.arange."""
    if stop is None:
        stop = start
        start = 0
    length = int(math.ceil((stop - start) / step))
    return np.array([start + i*step for i in range(length)], dtype=np.float64)

def manual_bincount(x, minlength=0):
    """Mengganti np.bincount: hitung frekuensi nilai integer."""
    if minlength == 0:
        minlength = int(max(x)) + 1 if x.size > 0 else 0
    hist = [0] * minlength
    for val in x.flat:
        hist[int(val)] += 1
    return np.array(hist, dtype=np.float64)

def manual_cumsum(arr):
    """Mengganti np.cumsum (kumulatif)."""
    res = np.zeros_like(arr)
    s = 0.0
    for i in range(arr.size):
        s += arr.flat[i]
        res.flat[i] = s
    return res

def manual_pad_reflect(img, pad):
    """Padding refleksi mirip np.pad(mode='reflect')."""
    h, w = img.shape
    padded = np.zeros((h + 2*pad, w + 2*pad), dtype=img.dtype)
    padded[pad:pad+h, pad:pad+w] = img
    # vertikal
    for i in range(pad):
        padded[i, pad:pad+w] = img[pad - i - 1, :]
        padded[h + pad + i, pad:pad+w] = img[h - i - 1, :]
    # horizontal
    for i in range(pad):
        padded[:, i] = padded[:, 2*pad - i - 1]
        padded[:, w + pad + i] = padded[:, w + pad - i - 1]
    return padded


# ============================================================
# 1. RESIZE MANUAL (bilinear, tanpa np.ix_, np.floor, np.clip)
# ============================================================
def resize_manual(img, ukuran_baru):
    tinggi_lama, lebar_lama = img.shape
    lebar_baru, tinggi_baru = ukuran_baru
    img_f = img.astype(np.float64)

    skala_x = lebar_lama / lebar_baru
    skala_y = tinggi_lama / tinggi_baru

    y_asal = manual_clip(
        (manual_arange(tinggi_baru) + 0.5) * skala_y - 0.5,
        0, tinggi_lama - 1
    )
    x_asal = manual_clip(
        (manual_arange(lebar_baru) + 0.5) * skala_x - 0.5,
        0, lebar_lama - 1
    )

    hasil = np.zeros((tinggi_baru, lebar_baru), dtype=np.float64)

    for i in range(tinggi_baru):
        y = y_asal[i]
        y0 = int(math.floor(y))
        y1 = min(y0 + 1, tinggi_lama - 1)
        wy = y - y0
        for j in range(lebar_baru):
            x = x_asal[j]
            x0 = int(math.floor(x))
            x1 = min(x0 + 1, lebar_lama - 1)
            wx = x - x0

            p00 = img_f[y0, x0]
            p01 = img_f[y0, x1]
            p10 = img_f[y1, x0]
            p11 = img_f[y1, x1]

            atas  = p00 * (1 - wx) + p01 * wx
            bawah = p10 * (1 - wx) + p11 * wx
            hasil[i, j] = atas * (1 - wy) + bawah * wy

    return manual_clip(hasil, 0, 255).astype(np.uint8)


# ============================================================
# 2. MEDIAN FILTER MANUAL (kernel 3x3)
# ============================================================
def median_filter_manual(img, kernel_size=3):
    pad = kernel_size // 2
    img_pad = manual_pad_reflect(img.astype(np.float64), pad)
    tinggi, lebar = img.shape
    hasil = np.zeros((tinggi, lebar), dtype=np.float64)

    for i in range(tinggi):
        for j in range(lebar):
            # ambil window
            window = []
            for ki in range(kernel_size):
                for kj in range(kernel_size):
                    window.append(img_pad[i+ki, j+kj])
            # urutkan (gunakan sorted bawaan Python, bukan numpy)
            window_sorted = sorted(window)
            median = window_sorted[len(window)//2]
            hasil[i, j] = median

    return hasil.astype(np.uint8)


# ============================================================
# 3. CLAHE MANUAL (tanpa np.meshgrid, np.bincount, np.cumsum, dll.)
# ============================================================
def clahe_manual(img, clip_limit=2.0, grid_size=(8, 8)):
    tinggi, lebar = img.shape
    gx, gy = grid_size
    tinggi_tile = tinggi // gy
    lebar_tile  = lebar // gx

    peta_tile = np.zeros((gy, gx, 256), dtype=np.float64)

    for ty in range(gy):
        y0 = ty * tinggi_tile
        y1 = tinggi if ty == gy - 1 else y0 + tinggi_tile
        for tx in range(gx):
            x0 = tx * lebar_tile
            x1 = lebar if tx == gx - 1 else x0 + lebar_tile

            tile = img[y0:y1, x0:x1]
            jumlah_piksel = tile.size

            hist = manual_bincount(tile.ravel(), minlength=256)
            batas = max(1.0, clip_limit * jumlah_piksel / 256.0)
            kelebihan = 0.0
            for i in range(256):
                if hist[i] > batas:
                    kelebihan += hist[i] - batas
                    hist[i] = batas
            hist += kelebihan / 256.0

            cdf = manual_cumsum(hist)
            peta_tile[ty, tx] = (cdf / cdf[-1]) * 255.0

    # interpolasi bilinear
    py = manual_clip(
        manual_arange(tinggi) / tinggi_tile - 0.5,
        0, gy - 1
    )
    px = manual_clip(
        manual_arange(lebar) / lebar_tile - 0.5,
        0, gx - 1
    )

    hasil = np.zeros((tinggi, lebar), dtype=np.float64)

    for i in range(tinggi):
        y = py[i]
        ty0 = int(math.floor(y))
        ty1 = min(ty0 + 1, gy - 1)
        wy = y - ty0

        for j in range(lebar):
            x = px[j]
            tx0 = int(math.floor(x))
            tx1 = min(tx0 + 1, gx - 1)
            wx = x - tx0

            val = img[i, j]
            v00 = peta_tile[ty0, tx0, val]
            v01 = peta_tile[ty0, tx1, val]
            v10 = peta_tile[ty1, tx0, val]
            v11 = peta_tile[ty1, tx1, val]

            atas  = v00 * (1 - wx) + v01 * wx
            bawah = v10 * (1 - wx) + v11 * wx
            hasil[i, j] = atas * (1 - wy) + bawah * wy

    return manual_clip(hasil, 0, 255).astype(np.uint8)


# ============================================================
# 4. OTSU THRESHOLDING MANUAL (tanpa numpy fungsi)
# ============================================================
def otsu_threshold_manual(img):
    """Menghitung threshold Otsu dan mengembalikan gambar biner (0 dan 255)."""
    hist = manual_bincount(img.ravel(), minlength=256)
    total_piksel = img.size
    # normalisasi histogram
    hist_norm = hist / total_piksel

    # hitung mean global
    mean_global = 0.0
    for i in range(256):
        mean_global += i * hist_norm[i]

    # variabel untuk best threshold
    best_thresh = 0
    best_var = 0.0

    # probabilitas dan mean kumulatif
    prob_k = 0.0
    mean_k = 0.0

    for t in range(256):
        prob_k += hist_norm[t]
        if prob_k == 0.0 or prob_k == 1.0:
            continue
        mean_k += t * hist_norm[t]  # ini sebenarnya mean kumulatif, tapi kita gunakan di dalam loop
        # variance between class
        var_between = (mean_global * prob_k - mean_k) ** 2 / (prob_k * (1 - prob_k))
        if var_between > best_var:
            best_var = var_between
            best_thresh = t

    # terapkan threshold
    binary = np.zeros_like(img)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            binary[i, j] = 255 if img[i, j] > best_thresh else 0
    return binary, best_thresh


# ============================================================
# 5. IMWRITE MANUAL (BMP 8-bit) — tetap sama
# ============================================================
def imwrite_manual(path, img):
    tinggi, lebar = img.shape
    img = img.astype(np.uint8)

    padding = (4 - (lebar % 4)) % 4
    ukuran_data = (lebar + padding) * tinggi
    offset_data = 14 + 40 + (256 * 4)
    ukuran_file = offset_data + ukuran_data

    # palet
    palet = np.zeros((256, 4), dtype=np.uint8)
    for i in range(256):
        palet[i, 0] = i
        palet[i, 1] = i
        palet[i, 2] = i

    if padding > 0:
        data_pad = np.zeros((tinggi, lebar + padding), dtype=np.uint8)
        data_pad[:, :lebar] = img
        data_piksel = data_pad[::-1].tobytes()
    else:
        data_piksel = img[::-1].tobytes()

    with open(path, 'wb') as f:
        f.write(b'BM')
        f.write(struct.pack('<I', ukuran_file))
        f.write(struct.pack('<H', 0))
        f.write(struct.pack('<H', 0))
        f.write(struct.pack('<I', offset_data))
        f.write(struct.pack('<I', 40))
        f.write(struct.pack('<i', lebar))
        f.write(struct.pack('<i', tinggi))
        f.write(struct.pack('<H', 1))
        f.write(struct.pack('<H', 8))
        f.write(struct.pack('<I', 0))
        f.write(struct.pack('<I', ukuran_data))
        f.write(struct.pack('<i', 0))
        f.write(struct.pack('<i', 0))
        f.write(struct.pack('<I', 256))
        f.write(struct.pack('<I', 256))
        f.write(palet.tobytes())
        f.write(data_piksel)
    return True


# ============================================================
# PROSES UTAMA (Grayscale → Resize → Median Filter → CLAHE → Otsu)
# ============================================================
print("Memulai proses preprocessing (Grayscale → Resize → Median Filter → CLAHE → Otsu) ...")

for label in KATEGORI:
    folder_input = os.path.join(PATH_INPUT, label)
    folder_output = os.path.join(PATH_OUTPUT, label)

    if not os.path.exists(folder_input):
        print(f"⚠️ Melewati folder (tidak ditemukan): {folder_input}")
        continue

    os.makedirs(folder_output, exist_ok=True)

    jumlah_sukses = 0
    for nama_file in tqdm(os.listdir(folder_input), desc=label):
        if nama_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            jalur_masuk = os.path.join(folder_input, nama_file)
            nama_dasar = os.path.splitext(nama_file)[0]
            jalur_keluar = os.path.join(folder_output, nama_dasar + ".bmp")

            # 1. Baca grayscale
            img = cv2.imread(jalur_masuk, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # 2. Resize ke 256x256
            img_resized = resize_manual(img, (256, 256))

            # 3. Median Filter (3x3)
            img_median = median_filter_manual(img_resized, kernel_size=3)

            # 4. CLAHE
            img_clahe = clahe_manual(img_median, clip_limit=2.0, grid_size=(8, 8))

            # 5. Otsu Thresholding -> binary (0/255)
            img_binary, _ = otsu_threshold_manual(img_clahe)

            # 6. Simpan
            imwrite_manual(jalur_keluar, img_binary)
            jumlah_sukses += 1

    print(f"✅ Selesai memproses {jumlah_sukses} gambar di folder: {label}")

print(f"\n🎉 Selesai! Hasil ada di:\n{PATH_OUTPUT}")

Memulai proses preprocessing (Grayscale → Resize → Median Filter → CLAHE → Otsu) ...


Normal:   0%|          | 0/100 [00:00<?, ?it/s]

Normal: 100%|██████████| 100/100 [00:50<00:00,  1.98it/s]


✅ Selesai memproses 100 gambar di folder: Normal


kidneyStone: 100%|██████████| 100/100 [00:49<00:00,  2.03it/s]

✅ Selesai memproses 100 gambar di folder: kidneyStone

🎉 Selesai! Hasil ada di:
D:/semester 4/PCD/praktikum/projek/test/Assets_Prepro4/
